# NETRA — Colab retrain

**Team Vortex · SIH 2026 · PS SIH26146**

Retrains both models on Colab and brings the artifacts back into the repository.

It is a **thin wrapper around the repository** — no logic lives here. Every step
calls the same `tasks.py` command the demo machine runs, so what Colab produces
and what the demo shows cannot drift apart.

---

## The one reason to retrain here rather than locally

**Colab ships `xgboost` preinstalled.** NETRA's explainability checks for it,
and when it is present the per-lead contributions switch from decision-path
(Saabas) values to **exact TreeSHAP** — the gold standard: game-theoretically
justified, correct under feature interactions, and summing exactly to the
prediction.

So this notebook does not just move the training somewhere faster. It **upgrades
the explanation method**, and the model card records which method produced the
artifacts.

---

## Read this before running anything

The demo target is **Python 3.10** with the versions pinned in
`requirements.txt`. Colab's runtime is a different Python and ships **numpy
2.x**.

Two rules decide whether the artifacts you train here will load on the demo
machine:

1. **`scikit-learn` must match exactly (1.6.1).** That is the version whose
   pickle format the demo machine reads.
2. **`numpy` must stay on the 1.x line.** A `.joblib` pickled under numpy 2.x
   **cannot** be loaded by numpy 1.x — it fails with
   `ModuleNotFoundError: No module named 'numpy._core'`.

The install cell below pins accordingly and the next cell **asserts** it. A
model is not portable just because the file copies — the library versions that
wrote it are part of the artifact.

Run the cells in order.

## Step 0 — what environment are we actually on?

Look before changing anything, so that if the install behaves unexpectedly you
can see what it was changing *from*.

In [ ]:
import platform, sys

print('=' * 62)
print('COLAB ENVIRONMENT (before pinning)')
print('=' * 62)
print('python  ', sys.version.split()[0])
print('platform', platform.platform())
print()
for name in ('numpy', 'pandas', 'sklearn', 'scipy', 'networkx', 'joblib', 'xgboost'):
    try:
        module = __import__(name)
        print(f'{name:10} {getattr(module, "__version__", "unknown")}')
    except ImportError:
        print(f'{name:10} (not installed)')
print('=' * 62)
print('xgboost present means the retrain will use EXACT TreeSHAP.')

## Step 1 — install the pinned stack

`numpy==1.23.5` from `requirements.txt` has no wheels for Colab's Python, so
this takes **the newest numpy 1.x line** instead. Pickles written under any
numpy 1.x are readable by every other numpy 1.x — that is the property that
matters, and it is why numpy 2.x is explicitly excluded.

In [ ]:
import subprocess, sys

PINS = [
    'numpy==1.26.4',          # newest 1.x with wheels for Colab's Python
    'pandas==2.1.4',          # pinned in requirements.txt
    'scipy==1.11.4',          # pinned; pulled in by scikit-learn
    'networkx==2.6.3',        # entity clustering, communities, centrality
    'scikit-learn==1.6.1',    # MUST match the demo machine exactly
    'joblib==1.2.0',          # writes the .joblib artifacts
    'jsonschema==4.17.3',     # contract validation
    'fastapi==0.120.0',
    'uvicorn==0.38.0',
    'python-multipart==0.0.20',
    'pytest==8.4.2',
    'httpx==0.28.1',
]

print('installing the pinned stack (about a minute)...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', *PINS],
    capture_output=True, text=True,
)
if result.stdout:
    print(result.stdout[-3000:])
if result.returncode != 0:
    print(result.stderr[-4000:])
    raise SystemExit('pip install failed -- read the error above')
print('install finished')

### If numpy changed version above

Downgrading numpy while the kernel is running can leave the old version in
memory. If the next cell reports a problem: **Runtime → Restart session**, then
continue from it. Nothing is lost — the clone and the training happen later.

In [ ]:
import importlib, sys

EXPECTED_SKLEARN = '1.6.1'

def version(name):
    return getattr(importlib.import_module(name), '__version__', 'unknown')

numpy_version = version('numpy')
sklearn_version = version('sklearn')
try:
    xgboost_version = version('xgboost')
except ImportError:
    xgboost_version = None

print('python      ', sys.version.split()[0])
print('numpy       ', numpy_version)
print('scikit-learn', sklearn_version)
print('pandas      ', version('pandas'))
print('xgboost     ', xgboost_version or 'NOT AVAILABLE')
print()

problems = []
if not numpy_version.startswith('1.'):
    problems.append('numpy is ' + numpy_version + ' -- 2.x artifacts do NOT load on the demo machine')
if sklearn_version != EXPECTED_SKLEARN:
    problems.append('scikit-learn is ' + sklearn_version + ', expected ' + EXPECTED_SKLEARN)

if problems:
    print('ENVIRONMENT IS NOT COMPATIBLE WITH THE DEMO TARGET:')
    for problem in problems:
        print('  -', problem)
    print()
    print('Runtime -> Restart session, re-run the install cell, then this one.')
    raise SystemExit('refusing to train in an incompatible environment')

print('environment is compatible with the demo target')
print('explanation method this run will use:',
      'exact TreeSHAP (xgboost)' if xgboost_version else 'decision-path contributions (Saabas)')

## Step 2 — get the repository

**Edit `REPO_URL` below.** If the repository is private, supply a GitHub token
when prompted (a fine-grained token with `Contents: Read and write` on this
repository). The input is hidden and used only for this session's git remote.

In [ ]:
import getpass, os, subprocess
from pathlib import Path

# ---------------------------------------------------------------------------
# EDIT THIS LINE
# ---------------------------------------------------------------------------
REPO_URL = 'https://github.com/YOUR-USERNAME/netra.git'
BRANCH = 'main'
# ---------------------------------------------------------------------------

WORK = Path('/content/netra')

if WORK.exists():
    print(f'{WORK} already exists -- fetching the latest')
    for args in (['fetch', '--all'], ['checkout', BRANCH], ['pull', '--ff-only']):
        subprocess.run(['git', '-C', str(WORK), *args], check=True)
else:
    url = REPO_URL
    token = getpass.getpass('GitHub token (hidden; press Enter if the repo is public): ').strip()
    if token:
        url = REPO_URL.replace('https://', 'https://' + token + '@', 1)
    print('cloning...')
    subprocess.run(['git', 'clone', '--branch', BRANCH, url, str(WORK)], check=True)

os.chdir(WORK)
print()
print('working directory:', Path.cwd())
for path in sorted(Path.cwd().iterdir()):
    if path.name != '.git':
        print('   ', path.name)

## Step 3 — generate the dataset

The dataset is **generated, never committed**: the generator is the source of
truth, and a fixed seed makes the output reproducible. The defaults reproduce
the dataset the README reports.

In [ ]:
!python tasks.py gen

## Step 4 — train, measure and write the artifacts

This fits the `IsolationForest` anomaly detector (unsupervised — it never sees a
label) and the `RandomForest` risk scorer, measures both with **batch-grouped
cross-validation**, runs the three experiments that answer *"is it just rules?"*,
and writes everything to `models/` — including the training distribution the
drift check compares against.

In [ ]:
!python tasks.py train

In [ ]:
import json
from pathlib import Path

metrics = json.loads(Path('models/metrics.json').read_text(encoding='utf-8'))
print('explanation method  :', metrics.get('explanation_method'))
print('cross-validation    :', metrics.get('cv_method'))
print('training mode       :', metrics.get('training_mode'))
print()
for key in ('cv_auc_mean', 'cv_auc_std', 'cv_precision_mean', 'cv_recall_mean',
            'cv_f1_mean', 'brier', 'cluster_ari'):
    print(f'  {key:22} {metrics.get(key)}')

## Step 5 — run the windowed pipeline

Splits the capture into batches, pins wallet identity across them, detects what
changed, and records the history the Monitoring page replays.

In [ ]:
!python tasks.py replay

## Step 6 — prove the artifacts reload, and record the environment

This is the compatibility guard. It loads the `.joblib` files **from disk**,
exactly as the demo server does, and runs the whole verification in one go. If
the pickle format or the pinned versions were wrong, it fails here — loudly,
now — instead of on stage.

`models/ENVIRONMENT.txt` is written alongside the artifacts, so if the demo
machine ever complains, the first question (*did the versions drift?*) is
answered by a file rather than by memory.

In [ ]:
!python tasks.py smoke

In [ ]:
from pathlib import Path

print(Path('models/ENVIRONMENT.txt').read_text(encoding='utf-8'))
print('artifact sizes:')
for name in ('risk.joblib', 'anomaly.joblib', 'metrics.json',
             'reference_distribution.json', 'feature_columns.json'):
    path = Path('models') / name
    if path.exists():
        print(f'   {name:32} {path.stat().st_size / 1024:8.1f} KB')
    else:
        print(f'   {name:32} MISSING')

## Step 7 — package the artifacts and download them

One zip containing the trained models, the drift reference, the measured
scorecard and the environment record.

In [ ]:
import shutil
from pathlib import Path

from google.colab import files

REPO = Path('/content/netra')
STAGE = Path('/content/netra_colab_artifacts')

shutil.rmtree(STAGE, ignore_errors=True)
(STAGE / 'models').mkdir(parents=True)

ARTIFACTS = [
    'models/risk.joblib',
    'models/anomaly.joblib',
    'models/feature_columns.json',
    'models/metrics.json',
    'models/ENVIRONMENT.txt',
    'models/reference_distribution.json',
]

for relative in ARTIFACTS:
    source = REPO / relative
    if source.exists():
        shutil.copy2(source, STAGE / relative)
        print('packed ', relative)
    else:
        print('MISSING', relative)

archive = shutil.make_archive('/content/netra_colab_artifacts', 'zip', STAGE)
print()
print('archive:', archive)
print(f'size   : {Path(archive).stat().st_size / 1024:.1f} KB')
print('downloading...')
files.download(archive)

## Step 8 — get the artifacts back into the repository

**Option A — copy them in locally (safest).** Unzip, copy `models/*` over the
matching folder, then confirm the demo machine agrees with Colab:

```bash
python tasks.py train      # or just: python tasks.py smoke
python tests/api_smoke.py  # the API serves the new analysis
```

Then commit the refreshed artifacts:

```bash
git add models/
git status --short         # the .joblib files SHOULD appear -- see .gitignore
git commit -m "Retrain on Colab: refresh model artifacts and metrics"
git push
```

The `.gitignore` deliberately lets `models/*.joblib`, `metrics.json` and
`reference_distribution.json` through: **a model is only reproducible if you ship
the exact bytes you measured.** Otherwise the README quotes numbers the shipped
`.joblib` no longer produces — and for an investigative tool, a number is
evidence, so the artifact and the claim have to travel together.

**Option B — push straight from Colab.** The final cell commits `models/` back
without a download. It needs a token with write access.

In [ ]:
import getpass, subprocess

PUSH = False   # set to True to commit the artifacts straight back

if not PUSH:
    print('PUSH is False -- nothing was sent. Flip it to True to commit models/ back.')
else:
    token = getpass.getpass('GitHub token with repo write access (hidden): ').strip()
    current = subprocess.run(['git', 'remote', 'get-url', 'origin'],
                             capture_output=True, text=True).stdout.strip()
    clean = 'https://' + current.split('@')[-1].split('https://')[-1]
    subprocess.run(['git', 'remote', 'set-url', 'origin',
                    clean.replace('https://', 'https://' + token + '@', 1)], check=True)
    subprocess.run(['git', 'config', 'user.name', 'Team Vortex'], check=True)
    subprocess.run(['git', 'config', 'user.email', 'team-vortex@example.com'], check=True)

    subprocess.run(['git', 'add', 'models/'], check=True)
    staged = subprocess.run(['git', 'diff', '--cached', '--name-only'],
                            capture_output=True, text=True).stdout.strip()
    print('staged for commit:')
    print(staged or '  (nothing changed -- the artifacts are byte-identical)')

    if staged:
        subprocess.run(['git', 'commit', '-m',
                        'Retrain on Colab: refresh model artifacts and metrics'], check=True)
        subprocess.run(['git', 'push'], check=True)
        print('pushed to origin')
    else:
        print('nothing to commit')

---

## If something goes wrong

| Symptom | Cause | Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'numpy._core'` | artifacts trained under numpy 2.x | re-run Step 1, restart the runtime, retrain |
| `InconsistentVersionWarning` on load | scikit-learn differs from 1.6.1 | re-run Step 1 and retrain |
| `No module named 'tasks'` | the kernel's working directory was lost | re-run the Step 2 clone cell |
| `pip` fails to resolve | Colab's preinstalled versions conflict with the pins | restart the runtime, then run Step 1 first |
| Clone asks for a password | private repository without a token | supply a fine-grained token with `Contents: Read and write` |
| `git push` rejected | the token lacks write scope | re-issue the token, `git pull --rebase` first |
| Metrics differ from the README | a different seed or dataset size | use the Step 3 defaults, or update the README — never leave the two disagreeing |

## What to tell a judge about this notebook

Colab is not where the product runs — the product runs **offline on one Linux
host**. Colab is where we **retrain and measure**, and the notebook exists so that
training is reproducible and reviewable rather than something that happened once
on somebody's laptop.

The interesting engineering content is the **version discipline**, plus the fact
that running here upgrades the explanations to exact TreeSHAP. A model is not
portable just because the file copies: the environment is pinned, asserted, and
recorded next to the artifact it produced.